# COVID-19 ML Micro Project

**Objective:** Explore COVID-19 data and use Linear Regression to predict confirmed cases from previous-day and previous-week case counts.

> Dataset file required: `covid19_ml_micro_project_raw.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("covid19_ml_micro_project_raw.csv")
df.head()

print("Dataset Shape:", df.shape)
print("Column Names:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)
print("Missing Values:")
print(df.isnull().sum())
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate rows after cleaning:", df.duplicated().sum())

print("Missing values before filling:")
print(df.isnull().sum())

numeric_columns = ["Deaths", "Recovered", "Active"]
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values after filling:")
print(df.isnull().sum())

# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Convert case counts to integers
integer_columns = ["Confirmed", "Deaths", "Recovered", "Active"]
for col in integer_columns:
    df[col] = df[col].astype(int)

print(df.dtypes)
print("Final Dataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
print("\nData Types:")
print(df.dtypes)

In [ ]:
df.describe()

print("Average Confirmed Cases:", round(df["Confirmed"].mean(), 2))
print("Average Deaths:", round(df["Deaths"].mean(), 2))
print("Average Recovered:", round(df["Recovered"].mean(), 2))
print("Average Active Cases:", round(df["Active"].mean(), 2))

In [ ]:
# Calculate Death Rate
df["Death_Rate"] = np.where(
    df["Confirmed"] > 0,
    (df["Deaths"] / df["Confirmed"]) * 100,
    0
)

# Calculate Recovery Rate
df["Recovery_Rate"] = np.where(
    df["Confirmed"] > 0,
    (df["Recovered"] / df["Confirmed"]) * 100,
    0
)

df.head()

print("Average Death Rate:",
      round(df["Death_Rate"].mean(), 2), "%")
print("Average Recovery Rate:",
      round(df["Recovery_Rate"].mean(), 2), "%")

In [ ]:
df["Month"] = df["Date"].dt.to_period("M")
monthly_cases = df.groupby("Month")["Confirmed"].sum()

plt.figure(figsize=(10, 5))
plt.plot(
    monthly_cases.index.astype(str),
    monthly_cases.values,
    marker="o"
)
plt.title("Monthly COVID-19 Confirmed Cases")
plt.xlabel("Month")
plt.ylabel("Confirmed Cases")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
monthly_deaths = df.groupby("Month")["Deaths"].sum()

plt.figure(figsize=(10, 5))
plt.plot(
    monthly_deaths.index.astype(str),
    monthly_deaths.values,
    marker="o"
)
plt.title("Monthly COVID-19 Deaths")
plt.xlabel("Month")
plt.ylabel("Deaths")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
country_cases = df.groupby("Country/Region")["Confirmed"].max()
country_cases = country_cases.sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(country_cases.index, country_cases.values)
plt.title("COVID-19 Confirmed Cases by Country")
plt.xlabel("Country")
plt.ylabel("Confirmed Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df = df.sort_values(["Country/Region", "Date"])

df["Previous_Day_Cases"] = df.groupby(
    "Country/Region"
)["Confirmed"].shift(1)

df["Previous_Week_Cases"] = df.groupby(
    "Country/Region"
)["Confirmed"].shift(7)

df[[
    "Date",
    "Country/Region",
    "Confirmed",
    "Previous_Day_Cases",
    "Previous_Week_Cases"
]].head(10)

In [ ]:
ml_data = df.dropna(
    subset=[
        "Previous_Day_Cases",
        "Previous_Week_Cases"
    ]
).copy()

print("ML Dataset Shape:", ml_data.shape)

In [ ]:
X = ml_data[
    ["Previous_Day_Cases", "Previous_Week_Cases"]
]
y = ml_data["Confirmed"]

split_date = ml_data["Date"].quantile(0.8)

train_data = ml_data[
    ml_data["Date"] <= split_date
]

test_data = ml_data[
    ml_data["Date"] > split_date
]

X_train = train_data[
    ["Previous_Day_Cases", "Previous_Week_Cases"]
]
y_train = train_data["Confirmed"]

X_test = test_data[
    ["Previous_Day_Cases", "Previous_Week_Cases"]
]
y_test = test_data["Confirmed"]

print("Training Records:", len(train_data))
print("Testing Records:", len(test_data))

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

In [ ]:
y_pred = model.predict(X_test)

print("Actual Values:")
print(y_test.head().values)

print("\nPredicted Values:")
print(y_pred[:5])

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", round(mae, 2))
print("Root Mean Squared Error:", round(rmse, 2))
print("R2 Score:", round(r2, 2))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    y_test.values,
    label="Actual Cases"
)
plt.plot(
    y_pred,
    label="Predicted Cases"
)
plt.title("Actual vs Predicted COVID-19 Cases")
plt.xlabel("Test Data Samples")
plt.ylabel("Confirmed Cases")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()